# Tableau Data Prep — Reddit Movie Pulse Dashboard

This notebook creates clean, Tableau-optimized CSV files from your existing data.

## 1. Load All Data

In [1]:
import pandas as pd
import numpy as np

# load everything
comparison = pd.read_csv("data/platform_comparison.csv")
movie_scores = pd.read_csv("data/reddit_movie_scores_v2.csv")
comments = pd.read_csv("data/reddit_sentiment_multilingual.csv")
comments["comment_timestamp"] = pd.to_datetime(comments["comment_timestamp"])

print(f"Comparison: {len(comparison)} movies")
print(f"Movie scores: {len(movie_scores)} movies")
print(f"Comments: {len(comments)} comments")

Comparison: 108 movies
Movie scores: 210 movies
Comments: 60378 comments


## 2. Create Tableau File 1 — Movie Level Comparison

One row per movie with all ratings, genres exploded for filtering.

In [2]:
# start from comparison data
tab_movies = comparison.copy()

# add v2 scores
v2_cols = ["movie_name", "reddit_score_v2", "multi_avg", "english_pct"]
v2 = movie_scores[v2_cols].copy()
tab_movies = tab_movies.merge(v2, on="movie_name", how="left")

# explode genres — Tableau works better with one genre per row for filtering
# but also keep original genre string
tab_movies["genre_original"] = tab_movies["genre"]
tab_movies["genre_list"] = tab_movies["genre"].str.split(", ")
tab_movies_exploded = tab_movies.explode("genre_list")
tab_movies_exploded = tab_movies_exploded.rename(columns={"genre_list": "genre_single"})

# add rating difference columns
tab_movies_exploded["reddit_minus_imdb"] = (tab_movies_exploded["reddit_score"] - tab_movies_exploded["imdb_rating"]).round(2)
tab_movies_exploded["reddit_minus_rt"] = (tab_movies_exploded["reddit_score"] - tab_movies_exploded["rt_score_10"]).round(2)
tab_movies_exploded["imdb_minus_rt"] = (tab_movies_exploded["imdb_rating"] - tab_movies_exploded["rt_score_10"]).round(2)

# rating agreement category
def agreement_level(diff):
    diff = abs(diff)
    if diff < 0.5:
        return "Strong Agreement"
    elif diff < 1.0:
        return "Moderate Agreement"
    elif diff < 2.0:
        return "Disagreement"
    else:
        return "Strong Disagreement"

tab_movies_exploded["reddit_imdb_agreement"] = tab_movies_exploded["reddit_minus_imdb"].apply(agreement_level)

print(f"Rows (with genre exploded): {len(tab_movies_exploded)}")
print(f"Columns: {tab_movies_exploded.columns.tolist()}")

tab_movies_exploded.to_csv("data/tableau_movies.csv", index=False)
print("Saved to data/tableau_movies.csv")

Rows (with genre exploded): 283
Columns: ['movie_name', 'year', 'genre', 'region', 'reddit_score', 'vader_avg', 'textblob_avg', 'roberta_avg', 'imdb_rating', 'rt_score_10', 'rt_score', 'total_comments', 'explicit_ratings', 'positive_pct', 'negative_pct', 'reddit_vs_imdb', 'reddit_vs_rt', 'popularity', 'reddit_score_v2', 'multi_avg', 'english_pct', 'genre_original', 'genre_single', 'reddit_minus_imdb', 'reddit_minus_rt', 'imdb_minus_rt', 'reddit_imdb_agreement']
Saved to data/tableau_movies.csv


## 3. Create Tableau File 2 — Comment Level Sentiment

One row per comment for sentiment deep-dives.

In [3]:
# keep useful columns, add some derived fields
tab_comments = comments[[
    "movie_name", "subreddit", "comment_text", "upvotes",
    "comment_timestamp", "extracted_rating",
    "multi_score", "multi_label",
    "language", "final_score_v2", "final_label_v2"
]].copy()

# time features for Tableau
tab_comments["comment_date"] = tab_comments["comment_timestamp"].dt.date
tab_comments["comment_month"] = tab_comments["comment_timestamp"].dt.to_period("M").astype(str)
tab_comments["comment_year"] = tab_comments["comment_timestamp"].dt.year
tab_comments["day_of_week"] = tab_comments["comment_timestamp"].dt.day_name()

# comment length bucket
tab_comments["comment_length"] = tab_comments["comment_text"].str.len()
tab_comments["length_bucket"] = pd.cut(
    tab_comments["comment_length"],
    bins=[0, 50, 150, 500, 1000],
    labels=["Short (<50)", "Medium (50-150)", "Long (150-500)", "Very Long (500+)"]
)

# has explicit rating flag
tab_comments["has_explicit_rating"] = tab_comments["extracted_rating"].notna().map({True: "Yes", False: "No"})

print(f"Rows: {len(tab_comments)}")
tab_comments.to_csv("data/tableau_comments.csv", index=False)
print("Saved to data/tableau_comments.csv")

Rows: 60378
Saved to data/tableau_comments.csv


## 4. Create Tableau File 3 — Monthly Aggregated Sentiment

Pre-aggregated for time series charts.

In [4]:
monthly = comments.groupby([
    comments["comment_timestamp"].dt.to_period("M").astype(str),
    "movie_name"
]).agg(
    avg_score = ("multi_score", "mean"),
    comment_count = ("multi_score", "count"),
    positive_count = ("multi_label", lambda x: (x == "positive").sum()),
    negative_count = ("multi_label", lambda x: (x == "negative").sum()),
    neutral_count = ("multi_label", lambda x: (x == "neutral").sum()),
    avg_upvotes = ("upvotes", "mean")
).reset_index()

monthly.columns = ["month", "movie_name", "avg_score", "comment_count",
                    "positive_count", "negative_count", "neutral_count", "avg_upvotes"]

monthly["positive_pct"] = (monthly["positive_count"] / monthly["comment_count"] * 100).round(1)

print(f"Rows: {len(monthly)}")
monthly.to_csv("data/tableau_monthly.csv", index=False)
print("Saved to data/tableau_monthly.csv")

Rows: 2540
Saved to data/tableau_monthly.csv


## 5. Create Tableau File 4 — Model Accuracy Comparison

For the sentiment model comparison charts.

In [5]:
# model comparison data
model_data = []

# overall accuracy from your earlier analysis
models = [
    {"model": "VADER", "mae": 2.65, "correlation": 0.125, "type": "Lexicon-based"},
    {"model": "TextBlob", "mae": 2.29, "correlation": 0.127, "type": "Lexicon-based"},
    {"model": "RoBERTa", "mae": 1.94, "correlation": 0.416, "type": "Transformer"},
    {"model": "Multilingual BERT", "mae": 1.93, "correlation": 0.365, "type": "Transformer"},
]

model_df = pd.DataFrame(models)
model_df.to_csv("data/tableau_model_accuracy.csv", index=False)
print("Saved model accuracy data")
print(model_df)

Saved model accuracy data
               model   mae  correlation           type
0              VADER  2.65        0.125  Lexicon-based
1           TextBlob  2.29        0.127  Lexicon-based
2            RoBERTa  1.94        0.416    Transformer
3  Multilingual BERT  1.93        0.365    Transformer


## 6. Summary

In [7]:
print("All Tableau files created:")
print("  1. data/tableau_movies.csv — movie-level comparison (connect as primary)")
print("  2. data/tableau_comments.csv — comment-level sentiment")
print("  3. data/tableau_monthly.csv — monthly aggregated trends")
print("  4. data/tableau_model_accuracy.csv — model accuracy comparison")
print()


All Tableau files created:
  1. data/tableau_movies.csv — movie-level comparison (connect as primary)
  2. data/tableau_comments.csv — comment-level sentiment
  3. data/tableau_monthly.csv — monthly aggregated trends
  4. data/tableau_model_accuracy.csv — model accuracy comparison

